# Dapur Domino — Eksplorasi Data, Desain Skenario, & Evaluasi Pemodelan Data Science
**Analisis Jaringan Efek Domino Harga Pangan Nasional & Sistem Rekomendasi Substitusi Bahan Kuliner Nusantara Sadar-Harga**

Notebook ini mendokumentasikan alur lengkap eksperimen *Data Science* secara end-to-end — mulai dari perumusan masalah bisnis & tujuan teknis, telaah eksplorasi data (EDA), validasi kualitas & seleksi objek data, pembersihan & rekayasa fitur bebas *data leakage*, pengujian skenario hyperparameter pada jendela validasi, pembangunan model deterministik, evaluasi *out-of-sample* beserta taksonomi error, hingga keputusan *review* pemodelan per komoditas.

---
## 1. Perumusan Objektif Bisnis & Tujuan Teknis Data Science

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSIS_DIR = REPO_ROOT / "artifacts" / "analysis"

business_df = pd.read_csv(ANALYSIS_DIR / "business_objectives.csv")
technical_df = pd.read_csv(ANALYSIS_DIR / "technical_objectives.csv")
display(business_df)
display(technical_df)

,objective_id,stakeholder,business_problem,business_goal,kpi_business,constraint
0,BO-1,Rumah Tangga & Konsumen Harian,"Saat harga komoditas bumbu/pangan melonjak, ko...",Menyediakan rekomendasi bahan pengganti yang m...,Cakupan bahan terekomendasi terhadap total pem...,"100% lokal (tanpa biaya API/cloud), berjalan d..."
1,BO-2,Pelaku UMKM Kuliner & Warung Makan,Kenaikan harga bahan pokok merambat antar-komo...,Memetakan jaringan efek domino harga antar-10 ...,Tingkat temuan palsu (false discovery) jaringa...,Hanya mengklaim sinyal pemicu jika terbukti me...
2,BO-3,Pengguna Dapur (Optimasi Bahan Tersedia),Bahan sisa di kulkas sering terbuang karena pe...,Merekomendasikan bahan pelengkap dan resep Nus...,Akurasi Top-5 prediksi bahan pelengkap mengala...,Waktu respons interaktif < 1 detik di antarmuk...
3,BO-4,Reviewer Teknis & Audit Riset Data Science,Banyak eksperimen sains data mencampur data la...,Menjamin ketertelusuran penuh (SHA-256 manifes...,"100% artefak terverifikasi SHA-256, 0% kebocor...",Reprodusibel penuh dengan seed terkunci (seed=...


,tech_objective_id,linked_business_id,ds_problem_type,metric_name,target_threshold,measured_result,status
0,TDO-1,"BO-1, BO-3",NLP Rule-Based & Dictionary Text Normalization,VP1 Clean Normalization Accuracy (n=300 eval s...,>= 90.0%,90.33% (271/300),TERCAPAI (VERIFIED)
1,TDO-2,BO-3,Graph Association (NPMI/Louvain) & Conditional...,VP2 Hide-One-Ingredient Top-5 Accuracy vs Popu...,> Popularity baseline with disjoint 95% Wilson CI,45.50% vs 40.88%,TERCAPAI (VERIFIED)
2,TDO-3,BO-1,Skip-Gram Word2Vec Embedding & Complement-Pena...,VP3 Substitution Recall@5 on 25 Culinary Refer...,Dilaporkan secara jujur beserta analisis error,24.0% (6/25),TERUKUR & DIEVALUASI JUJUR
3,TDO-4,BO-2,Time-Series Log-Return Stationarity (ADF + KPSS),VP4 Joint Stationarity Pass Rate (ADF p<0.05 &...,Mayoritas (>85%) seri lulus uji ganda,"Nas 90.0%, Prov 98.24%, Kota 98.00%",TERCAPAI (VERIFIED)
4,TDO-5,BO-2,Granger Causality Network + BH-FDR + Permutati...,VP5 False Edge Rate (20x Circular Shift) & VP7...,VP5 <= 5.0% FDR alpha; VP7 dilaporkan,VP5=3.44% (<=5%); VP7=100% (50/50 bertahan),TERCAPAI (VERIFIED)
5,TDO-6,"BO-2, BO-4",Out-of-Sample Walk-Forward ARX(5) Ridge vs AR(...,VP6 Out-of-Sample RMSE & Diebold-Mariano Test ...,Validasi selektif per komoditas (tanpa klaim b...,"2/10 lolos penuh di Raw (Daging Sapi, Minyak G...",TERCAPAI SELEKTIF (VERIFIED)
6,TDO-7,"BO-1, BO-4",Multi-Criteria Price-Aware Substitution Scoring,FR-C4 Recipe Mention Coverage & Unmapped Honesty,Cakupan kemunculan resep >= 25%; 0 harga fiktif,27.74% kemunculan resep (217.380/783.564),TERCAPAI (VERIFIED)


### Analisis Objektif Bisnis & Tujuan Teknis
- **Empat Objektif Bisnis (`BO-1` s.d. `BO-4`)** dipetakan menjadi **tujuh Tujuan Teknis Data Science (`TDO-1` s.d. `TDO-7`)** beserta jenis masalah statistik/ML (normalisasi NLP berbasis aturan, graf asosiasi & deteksi komunitas, *embedding* Skip-Gram, uji stasioneritas ganda, jaringan kausalitas Granger, *walk-forward forecasting*, dan skoring hibrida).
- Setiap metrik evaluasi, baseline pembanding, dan ambang kelulusan ditetapkan sebelum menguji data *holdout* untuk menjaga validitas eksperimen.
- Seluruh pipeline dijalankan 100% lokal dengan batas puncak penggunaan memori `<= 8 GB RAM` serta kebijakan tegas menolak estimasi harga fiktif bagi bahan yang tidak terpantau oleh Bank Indonesia.

---
## 2. Telaah Data (*Exploratory Data Analysis*)

In [2]:
eda_prices = pd.read_csv(ANALYSIS_DIR / "eda_prices_stats.csv")
eda_corr = pd.read_csv(ANALYSIS_DIR / "eda_returns_corr.csv")
eda_recipes = pd.read_csv(ANALYSIS_DIR / "eda_recipes_stats.csv")

display(eda_prices)
display(eda_recipes)

fig_cv = px.bar(
    eda_prices,
    x="commodity_name",
    y="cv_pct",
    color="return_std_daily_pct",
    title="Koefisien Variasi Harga (%) & Volatilitas Return Harian 10 Komoditas PIHPS (796 Hari Kerja)",
    labels={
        "commodity_name": "Komoditas",
        "cv_pct": "Koefisien Variasi (%)",
        "return_std_daily_pct": "Std Return Harian (%)",
    },
)
fig_cv.show()

corr_matrix = eda_corr.set_index("commodity_name")
fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-0.5,
    zmax=0.5,
    title="Matriks Korelasi Kontemporer Log-Return Harian Antar-10 Komoditas Nasional",
)
fig_corr.show()

,comcat_id,commodity_name,n_days,mean_idr,std_idr,cv_pct,min_idr,p25_idr,median_idr,p75_idr,max_idr,iqr_idr,return_mean_daily_pct,return_std_daily_pct,return_skewness,return_kurtosis,min_daily_return_pct,max_daily_return_pct
0,cat_8,Cabai Rawit,796,61034.99,12692.81,20.80,38800.0,51237.5,58975.0,69412.5,95700.0,18175.0,0.0358,2.1875,1.0393,7.7078,-11.13,13.90
1,cat_7,Cabai Merah,796,54101.26,10794.47,19.95,30950.0,47637.5,53275.0,59375.0,85100.0,11737.5,0.0443,2.5573,0.9644,15.7019,-16.98,22.07
2,cat_5,Bawang Merah,796,42186.12,6820.43,16.17,27250.0,37387.5,42100.0,47000.0,58100.0,9612.5,0.0441,1.3671,4.1677,56.0659,-7.95,19.33
3,cat_9,Minyak Goreng,796,20713.76,1392.12,6.72,18100.0,19200.0,20950.0,21250.0,23150.0,2050.0,0.0259,0.2723,1.1080,125.0929,-3.79,4.29
4,cat_6,Bawang Putih,796,41839.45,2614.99,6.25,37400.0,39700.0,41300.0,43950.0,48100.0,4250.0,0.0018,0.5547,2.0566,33.7049,-4.93,5.76
5,cat_2,Daging Ayam,796,38049.31,2227.07,5.85,30700.0,36187.5,37700.0,39300.0,43900.0,3112.5,0.0108,1.0775,2.0749,152.6487,-15.60,17.99
6,cat_4,Telur Ayam,796,30471.98,1186.83,3.89,26400.0,29550.0,30200.0,31450.0,33700.0,1900.0,0.0024,0.6373,-1.5935,168.5551,-10.76,9.65
7,cat_3,Daging Sapi,796,138706.72,4780.91,3.45,127900.0,135300.0,136800.0,140550.0,149900.0,5250.0,0.0133,0.3720,-1.8647,48.5795,-4.88,2.81
8,cat_10,Gula Pasir,796,18855.59,635.32,3.37,16450.0,18500.0,18900.0,19300.0,20150.0,800.0,0.0255,0.2385,0.4473,26.5478,-2.11,2.37
9,cat_1,Beras,796,15597.86,461.49,2.96,13750.0,15300.0,15700.0,15950.0,16550.0,650.0,0.0153,0.4206,0.6953,146.5770,-6.34,6.68


,aspect,count,mean,std,min,p25,median,p75,p95,max,note
0,"Jumlah Bahan Bersih per Resep (Train, n=66.419)",66419,13.0200,5.0000,1.00,9.000,13.0000,16.0000,22.00,67.0000,Rata-rata resep Nusantara memakai ~13 bahan un...
1,"Jumlah Bahan Bersih per Resep (Eval, n=3.496)",3496,13.0600,4.9200,1.00,10.000,13.0000,16.0000,22.00,36.0000,Distribusi split Eval identik dengan Train (ti...
2,Frekuensi Resep per Bahan di Kosakata (n=1.334...,1334,587.3800,3286.0400,20.00,29.000,46.0000,108.0000,1341.20,52972.0000,Distribusi power-law (Zipf): Top-10 bahan meny...
3,Bobot Asosiasi NPMI pada Graf Bahan (n=11.938 ...,11938,0.1518,0.1184,0.05,0.077,0.1139,0.1801,0.38,0.9391,"Pasangan bahan dengan NPMI >= 0,05 dan ko-okur..."
4,Ukuran Komunitas Rasa Louvain (n=82 komunitas),82,16.2700,69.1800,1.00,1.000,1.0000,1.0000,100.60,475.0000,"Terbentuk klaster bumbu dasar, kue/baking, rem..."


### Temuan Utama Eksplorasi Data (EDA)
- **Dispersi & Volatilitas Harga (`796` hari kerja, `1.141.526` catatan PIHPS):** Kelompok hortikultura berfluktuasi hampir 7 kali lebih tajam dibanding pangan pokok kering. **Cabai Rawit** mencatat koefisien variasi tertinggi (`CV = 20,80%`, rentang `Rp 38.800`–`Rp 95.700`, simpangan baku *return* harian `2,19%`), diikuti **Cabai Merah** (`CV = 19,95%`, simpangan baku `2,56%/hari`) dan **Bawang Merah** (`CV = 16,17%`, *skewness* positif `4,17`). Sebaliknya, **Beras** (`CV = 2,96%`), **Gula Pasir** (`CV = 3,37%`), dan **Daging Sapi** (`CV = 3,45%`) bergerak jauh lebih landai.
- **Kesetaraan Split & Distribusi *Power-Law* Resep (`66.419` train & `3.496` eval):** Jumlah rata-rata bahan per resep identik antara set latih (`13,02`, median `13`, IQR `7`) dan set uji (`13,06`, median `13`, IQR `6`), memastikan ketiadaan *covariate shift*. Di tingkat kosakata (`1.334` bahan), sebaran frekuensi mengikuti hukum *power-law* (Zipf): **Top-10 bahan menyumbang `40,64%`** dan **Top-50 menyumbang `80,40%`** seluruh kemunculan bahan di dapur.

---
## 3. Validasi Kualitas, Seleksi Objek Data, Pembersihan, & Rekayasa Fitur

In [3]:
val_df = pd.read_csv(ANALYSIS_DIR / "data_validation_report.csv")
sel_df = pd.read_csv(ANALYSIS_DIR / "data_selection_matrix.csv")
cln_df = pd.read_csv(ANALYSIS_DIR / "data_cleaning_audit.csv")
fea_df = pd.read_csv(ANALYSIS_DIR / "feature_engineering_audit.csv")

display(val_df)
display(sel_df)
display(cln_df)
display(fea_df)

,validation_check,scope,before_or_raw,validated_result,validation_status,action_taken
0,Integritas Kriptografis Data Mentah (SHA-256 M...,379 file mentah di data/raw/,379 file tercatat,0 error integritas (100% cocok),VALID,Setiap file diverifikasi ukuran byte & SHA-256...
1,Kelengkapan Kalender Hari Kerja & Hari Libur N...,Panel Harga PIHPS (2023-11-01 s.d. 2026-11-18),804 hari kerja Senin-Jumat,796 hari pasar buka (8 hari libur >=90% kosong...,VALID,Akhir pekan tidak diimputasi; celah pendek <=3...
2,Kelengkapan Seri Harga per Wilayah (Batas Miss...,"Nasional (10), Provinsi (340), Kota/Kab (1.099...",Total 1.449 seri komoditas x wilayah,"Nasional 10/10, Provinsi 340/340, Kota 1049/10...",VALID,50 seri kota yang jarang melapor dikeluarkan a...
3,Audit Anomali Lonjakan Harian (|ln(P_t/P_{t-1}...,1.399 seri harga bersih x 796 hari kerja,Pemeriksaan lonjakan ekstrem > ~91% sehari,"Nasional: 0 titik, Provinsi: 9 titik, Kota: 66...",VALID (TERCATAT),Dicatat di outlier_audit_*.csv tanpa memalsuka...
4,Validasi Stasioneritas Deret Waktu (Uji Ganda ...,Harga Mentah (Level) vs Log-Return pada 1.399 ...,"Level lulus ganda: Nas 0/10, Prov 68/340, Kota...","Log-return lulus ganda: Nas 9/10 (90.0%), Prov...",VALID,Pemodelan Granger & ARX wajib memakai log-retu...
5,Audit Kualitas Normalisasi Teks Bahan (VP1),300 baris acak dari eval.parquet,"Teks mentah berisi angka, satuan, kurung, sing...",271/300 (90.33%) bersih sempurna (>= ambang 90%),VALID,29 baris residu didokumentasikan untuk perbaik...


,data_object,selection_decision,volume_selected,rationale
0,Sumber Harga Harian (D1: PIHPS Bank Indonesia),DIPILIH (Utama),1.141.526 observasi (10 komoditas x 796 hari k...,Satu-satunya sumber resmi harian yang mencakup...
1,Sumber Indeks Bulanan (D2: Open Data Bapanas),PEMBANDING (Tidak digabung ke deret harian),130 baris sampel indikator bulanan,Berfrekuensi bulanan sehingga tidak cocok untu...
2,Seri Kota/Kabupaten dengan Missingness > 20%,DIEKSKLUSI (50 dari 1.099 seri kota),1049 seri kota dipertahankan,Mencegah distorsi autokorelasi akibat forward-...
3,Pembagian Waktu Deret Harga (Chronological Split),"Train [0:400], Val [400:500], Test [500:796]",500 hari pengembangan (400 sub-train + 100 val...,Menjaga urutan kausal waktu (0% look-ahead bia...
4,Korpus Resep Masakan (D3: junwatu/indonesian-r...,DIPILIH (Train 66.419 + Eval 3.496 resep),69.915 resep (897.519 baris bahan mentah),"Ukuran besar, berbahasa Indonesia asli, dan su..."
5,Kosakata Bahan Resep (Filter min_ingredient_fr...,DIPILIH 1.334 bahan kanonik,"Mencakup 783,564 dari 852,568 baris (91.91% to...",Memangkas salah ketik langka (<20 resep) sambi...


,cleaning_stage,raw_input_issue,cleaning_rule,output_impact
0,1. Parsing Angka & Tanggal PIHPS (ingest/pihps...,"Harga berformat string '58,450' atau '-', spas...","Hapus koma ribuan, konversi '-' ke NaN, strip ...","1,141,526 record bersih terstruktur di long pa..."
1,2. Penyelarasan Hari Kerja & Hari Libur (clean...,Hari libur nasional seluruh pasar kosong (NaN ...,Reindex ke frekuensi 'B' (Senin-Jumat) & drop ...,Dari 804 hari kerja menjadi 796 hari pasar buka.
2,3. Penanganan Missing Value Terbatas (cleaning...,Celah 1-3 hari pada pelaporan kota & seri pasi...,"Drop seri missing >20%, lalu forward-fill maks...",1049 seri kota bersih; 50 seri pasif dibuang.
3,4. Pembersihan Kuantitas & Satuan Resep (clean...,Teks seperti '250 gr dada ayam fillet (potong ...,"Hapus isi kurung, angka/pecahan, 80+ satuan da...","Memproses 852,568 baris latih & 44,951 baris uji."
4,5. Pemisahan Bahan Majemuk & Kamus Sinonim (in...,"Baris gabungan 'garam dan gula secukupnya', 'b...","Split konjungsi (, / & dan atau) + pemetaan 23...",Menghasilkan 1.334 bahan kanonik dengan akuras...


,feature_group,mathematical_formula,dimensionality,purpose_and_leakage_control
0,Log-Return Harian (r_t),r_t = ln(P_t / P_{t-1}),"796 hari x (10 Nas, 340 Prov, 1.049 Kota)",Menstasionerkan deret harga dan menyamakan ska...
1,Matriks Desain Musiman & Hari Raya (X_cal),"[1, DOW_1..4, sin(2pi*k*d/365.25), cos(2pi*k*d...",796 hari x 11 fitur eksogen,Koefisien OLS beta di-fit HANYA pada data lati...
2,Matriks Lag Autoregressive Exogenous ARX(p),"[1, y_{t-1}..y_{t-p}, x^{(1)}_{t-1}..x^{(m)}_{...",1 + p*(1 + m_triggers) fitur per komoditas,Hanya memakai nilai masa lalu (t-1 s.d. t-p) u...
3,"Bobot Graf Ko-okurensi Bahan (PMI, NPMI, Lift)","NPMI(a,b) = log(P(a,b)/(P(a)P(b))) / (-log P(a...","1.334 simpul, 11.938 sisi berarah/tak-berarah",Dihitung murni dari 66.419 resep train.parquet...
4,Vektor Embedding Bahan Word2Vec & Skor Substitusi,"sub(a,b) = cos(v_a, v_b) * (1 - max(0, NPMI(a,...",1.334 bahan x 64 dimensi vektor laten,Menangkap kesamaan peran kuliner (cosine tingg...
5,Skor Stabilitas Harga 30 Hari & Skor Hibrida,"S_price = exp(-max(0, dP_30d)) * exp(-5 * vol_...",10 komoditas x 140 wilayah (Nasional/Prov/Kota),"Bobot 0,6 / 0,4 dikunci di konfigurasi sebelum..."


### Analisis Validasi, Seleksi, Pembersihan, & Rekayasa Fitur
- **Validasi Kualitas & Stasioneritas Data:** Seluruh `370` berkas mentah terverifikasi keutuhannya (`0` error SHA-256). Pengujian stasioneritas ganda (`ADF p < 0,05` dan `KPSS p >= 0,05`) membuktikan harga mentah (*level*) non-stasioner (`0/10` lulus di tingkat nasional), sedangkan transformasi *log-return* menaikkan kelulusan ke **`90,0% (9/10)`** Nasional (`100%` ADF), **`98,24% (334/340)`** Provinsi, dan **`98,00% (1.028/1.049)`** Kota/Kabupaten.
- **Seleksi Objek Data & Pembersihan:** Dari `1.099` seri harga tingkat kota, `50` seri dengan *missingness* `> 20%` dieksklusi agar tidak menimbulkan autokorelasi buatan akibat *forward-fill*. Delapan hari libur nasional (`>= 90%` seri kosong) disaring tanpa mengimputasi akhir pekan. Di korpus resep, ambang frekuensi `>= 20` memilih `1.334` bahan kanonik yang mempertahankan **`91,91%` (`783.564 / 852.568`)** seluruh kemunculan bahan.
- **Rekayasa Fitur Bebas Kebocoran (*Zero Test Leakage*):** Pembagian waktu deret harga dikunci ke tiga tahap (**Sub-Train `[0:400]`**, **Validasi `[400:500]`**, **Sealed Test `[500:796]`**). Koefisien OLS kalender (`11` fitur hari-dalam-minggu, harmonik Fourier $k=1,2$, & jendela H-14 s.d. H+7 Hari Raya) dipasang murni pada data latih sebelum diterapkan ke rentang uji.

---
## 4. Desain Skenario Model & Profil Pembangunan Model

In [4]:
scen_ts = pd.read_csv(ANALYSIS_DIR / "scenarios_timeseries_val.csv")
scen_br = pd.read_csv(ANALYSIS_DIR / "scenarios_bridge_sensitivity.csv")
reg_df = pd.read_csv(ANALYSIS_DIR / "model_registry.csv")

display(scen_ts)
display(scen_br)
display(reg_df)

fig_scen = px.bar(
    scen_ts,
    x="scenario_id",
    y=["mean_skill_vs_ar_pct", "mean_skill_vs_naive_pct"],
    barmode="group",
    title="Perbandingan 8 Skenario Model Deret Waktu pada Jendela Validasi Hari [400:500]",
    labels={"value": "Rata-rata Peningkatan Skill (%)", "scenario_id": "ID Skenario"},
)
fig_scen.show()

,scenario_id,scenario_description,validation_window,lag_order,ridge_alpha,significant_edges_subtrain,mean_rmse_naive,mean_rmse_ar,mean_rmse_domino,mean_skill_vs_ar_pct,mean_skill_vs_naive_pct,commodities_beating_ar,commodities_vp6_valid_on_val
0,S01,AR(1) vs Domino ARX(1) | Ridge alpha=1.0 (Raw),Hari [400:500] (100 hari kerja validasi),1,1.000,19,0.006128,0.005876,0.006110,-1.264,-4.195,4,2
1,S02,AR(3) vs Domino ARX(3) | Ridge alpha=1.0 (Raw),Hari [400:500] (100 hari kerja validasi),3,1.000,48,0.006128,0.005923,0.006046,5.299,-2.000,6,1
2,S03,AR(5) vs Domino ARX(5) | OLS Nyaris Tanpa Regu...,Hari [400:500] (100 hari kerja validasi),5,0.001,43,0.006128,0.005873,0.005838,1.060,-6.642,5,2
3,S04,AR(5) vs Domino ARX(5) | Ridge Ringan alpha=0....,Hari [400:500] (100 hari kerja validasi),5,0.100,43,0.006128,0.005873,0.005780,5.978,-0.679,6,2
4,S05,AR(5) vs Domino ARX(5) | Ridge Standar alpha=1...,Hari [400:500] (100 hari kerja validasi),5,1.000,43,0.006128,0.005873,0.005994,4.656,-1.605,6,2
5,S06,AR(5) vs Domino ARX(5) | Ridge Kuat alpha=10.0...,Hari [400:500] (100 hari kerja validasi),5,10.000,43,0.006128,0.005873,0.006136,3.591,-2.516,6,3
6,S07,AR(3) vs Domino ARX(3) | Ridge alpha=1.0 (Seas...,Hari [400:500] (100 hari kerja validasi),3,1.000,53,0.006823,0.006440,0.006669,4.432,1.004,7,2
7,S08,AR(5) vs Domino ARX(5) | Ridge Standar alpha=1...,Hari [400:500] (100 hari kerja validasi),5,1.000,50,0.006823,0.006290,0.006597,4.164,1.586,7,4


,scenario_id,w_similarity,w_price,description,targets_evaluated,targets_with_mapped_candidate,mean_best_mapped_recipe_sim,mean_best_mapped_price_stability,harmonic_balance_index
0,BW-1,1.0,0.0,Murni Kemiripan Resep (Abaikan Harga),14,5,0.8248,0.9378,0.8777
1,BW-2,0.8,0.2,Dominan Resep (80% Resep / 20% Harga),14,5,0.8248,0.9378,0.8777
2,BW-3,0.6,0.4,Seimbang Sadar-Harga (60% Resep / 40% Harga - ...,14,5,0.8248,0.9378,0.8777
3,BW-4,0.5,0.5,Bobot Setara (50% Resep / 50% Harga),14,5,0.8248,0.9378,0.8777
4,BW-5,0.3,0.7,Dominan Harga (30% Resep / 70% Harga),14,5,0.8248,0.9378,0.8777


,model_id,model_name,algorithm_family,key_hyperparameters,training_data_scope,seed,duration_sec,peak_ram_mb
0,MDL-B1,Panel Log-Return & Dual Stationarity Screener ...,Econometric Time-Series Transformation & Hypot...,"alpha=0.05, min_obs=30, max_ffill_gap=3",1.399 seri harga x 796 hari kerja,42,50.80,796.10
1,MDL-B2,Calendar-Fourier-Holiday OLS Seasonal Residual...,Ordinary Least Squares (Linear Regression via ...,"fourier_harmonics=2, pre_holiday=14d, post_hol...",Fit khusus pada 500 hari kerja pertama (train[...,42,54.17,202.10
2,MDL-B3,Directed Granger Causality Network + Benjamini...,VAR/OLS Nested F-Test + Multiple Testing FDR C...,"granger_lag=5, xcorr_max_lag=10, fdr_alpha=0.0...",90 pasangan nasional + 11.220 pasangan prov + ...,42,54.17,202.10
3,MDL-B4,Expanding-Window Domino ARX(5) Ridge Forecaster,L2-Regularized Autoregressive Exogenous Linear...,"lag=5, ridge_alpha=1.0, horizon=1 business day",Expanding window dari hari ke-500 s.d. 796 (29...,42,54.17,202.10
4,MDL-A1,NPMI Co-occurrence Graph & Louvain Flavor Comm...,Unsupervised Graph Learning & Modularity Optim...,"min_freq=20, min_pair=10, min_npmi=0.05, resol...","66.419 resep latih -> 1.334 simpul, 11.938 sis...",42,52.23,313.57
5,MDL-A2,Conditional Co-occurrence PMI Complement Recom...,Probabilistic Information-Theoretic Ranking,"score = log P(c) + mean_{o in S} PMI(c, o)",66.419 resep latih (diuji pada 3.459 resep eval),42,52.23,313.57
6,MDL-A3,Skip-Gram Word2Vec + Complement-Penalized Subs...,Neural Word Representation Learning (Skip-Gram...,"dim=64, window=10, sg=1, negative=10, epochs=1...",66.419 resep latih (kosakata 1.334 bahan),42,52.23,313.57
7,MDL-C1,Price-Aware Hybrid Substitution Scorer,Multi-Criteria Exponential Volatility & Simila...,"w_sim=0.6, w_price=0.4, lookback=30 business days",14 bahan terpetakan x 10 komoditas BI x 140 wi...,42,0.13,92.90


### Analisis Eksperimen Skenario & Registri Model
- **Mengapa Skenario `S08` Dipilih (Jendela Validasi `Hari [400:500]`):**
  - `S01` (`lag=1`, `alpha=1,0`, Raw) terlalu pendek menangkap jeda rambatan pasar sehingga rata-rata *skill vs AR* negatif (`-1,264%`).
  - `S03` (`lag=5`, `alpha=0,001` mendekati OLS tanpa penalti) mengalami *overfitting* berat terhadap *Naive* (`-6,642%`) karena rasio parameter terhadap jumlah sampel membesar.
  - **`S08` (`lag=5`, `ridge_alpha=1,0`, *Seasonally Adjusted*)** mencatat keseimbangan terbaik dengan keunggulan rata-rata positif baik atas **AR(5) (`+4,164%`, menang di `7/10` komoditas)** maupun **Naive (`+1,586%`, lulus di `4/10` komoditas validasi)**.
- **Efisiensi Model Deterministik (`seed=42`):** Delapan komponen pemodelan (`MDL-B1`..`B4`, `MDL-A1`..`A3`, `MDL-C1`) beroperasi dalam satu lingkungan laptop dengan puncak memori tertinggi hanya **`796,10 MB`** dan waktu total eksekusi pipeline di bawah 3 menit.

---
## 5. Evaluasi *Out-of-Sample*, Taksonomi Error, & Keputusan *Review* Model

In [5]:
err_df = pd.read_csv(ANALYSIS_DIR / "vp3_error_taxonomy.csv")
rev_df = pd.read_csv(ANALYSIS_DIR / "model_review_decisions.csv")

display(err_df)
display(rev_df)

fig_rev = px.bar(
    rev_df,
    x="commodity_name",
    y=["skill_vs_ar_raw_pct", "skill_vs_ar_seasonal_pct"],
    barmode="group",
    color_discrete_sequence=["#b45309", "#15803d"],
    title="Evaluasi Out-of-Sample (296 Hari Uji) & Keputusan Model Produksi per Komoditas",
    labels={"commodity_name": "Komoditas", "value": "Skill vs AR(5) (%)"},
)
fig_rev.show()

,target_ingredient,expected_substitutes,top3_predicted,hit_at_5,first_hit_rank,reciprocal_rank,ndcg_at_5,ndcg_at_10,error_category,diagnostic_note
0,cabai merah besar,"cabai hijau, cabai merah, cabai merah keriting...","cabai merah, cabai hijau keriting, cabai merah...",True,1.0,1.00,0.4852,0.4852,HIT (Tepat di Top-5),Kandidat acuan 'cabai merah' ditemukan pada pe...
1,cabai merah keriting,"cabai hijau, cabai merah, cabai merah besar, c...","cabai merah, cabai merah gede, cabai merah besar",True,1.0,1.00,0.5087,0.5087,HIT (Tepat di Top-5),Kandidat acuan 'cabai merah' ditemukan pada pe...
2,cabai rawit,"cabai merah, cabai merah keriting, cabai rawit...","cabai rawit merah, cabai rawit setan, cabai ra...",True,1.0,1.00,0.3392,0.3392,HIT (Tepat di Top-5),Kandidat acuan 'cabai rawit merah' ditemukan p...
3,bawang bombay,"bawang merah, bawang putih, daun bawang","cuka putih, kaldu daging, jamur shiitake",False,NaN,0.00,0.0000,0.0000,Tetangga Konteks Resep Serumpun (Bukan Substit...,Kandidat teratas ('cuka putih') muncul di jeni...
4,bawang merah,"bawang bombay, bawang putih, daun bawang","ajinomoto, bawang merah bawang putih, di",False,NaN,0.00,0.0000,0.0000,Varian Potongan/Bentuk Bahan Sama,Tetangga terdekat ('ajinomoto') masih merupaka...
5,kemiri,"almond, kacang mete, kacang tanah, wijen","miri, lombok keriting, kunci",False,NaN,0.00,0.0000,0.0000,Tetangga Konteks Resep Serumpun (Bukan Substit...,Kandidat teratas ('miri') muncul di jenis masa...
6,santan,"kelapa parut, krimer nabati, susu cair, susu e...","santan dari kelapa, sun, santan dr kelapa",False,NaN,0.00,0.0000,0.0000,Varian Potongan/Bentuk Bahan Sama,Tetangga terdekat ('santan dari kelapa') masih...
7,gula merah,"gula pasir, kecap manis, madu, molasses","blok gula merah, bulatan gula merah, bulat gul...",False,NaN,0.00,0.0000,0.0000,Varian Potongan/Bentuk Bahan Sama,Tetangga terdekat ('blok gula merah') masih me...
8,gula pasir,"gula merah, madu, sirup","kaldu bubuk non msg, gula penyedap, kaldu peny...",False,NaN,0.00,0.0000,0.0000,Tetangga Konteks Resep Serumpun (Bukan Substit...,Kandidat teratas ('kaldu bubuk non msg') muncu...
9,mentega,"margarin, minyak goreng, minyak kelapa, minyak...","perasan jeruk lemon, mustard, margarin untuk m...",False,NaN,0.00,0.0000,0.0000,Tetangga Konteks Resep Serumpun (Bukan Substit...,Kandidat teratas ('perasan jeruk lemon') muncu...


,comcat_id,commodity_name,rmse_naive_raw,rmse_ar5_raw,rmse_domino_raw,skill_vs_ar_raw_pct,skill_vs_ar_seasonal_pct,vp6_raw_pass,vp6_seasonal_pass,review_decision,recommended_production_model,review_rationale,next_iteration_action
0,cat_1,Beras,0.001284,0.001245,0.001283,-3.01,6.62,False,True,DEPLOY_DOMINO_ARX5_SEASONAL (Menang Pasca-Kont...,Domino ARX(5) Ridge (Seasonal-Adjusted),"Setelah efek kalender/Lebaran dibersihkan, Dom...",Gunakan model dengan residualisasi musiman aktif.
1,cat_10,Gula Pasir,0.001613,0.001446,0.001596,-10.34,-8.50,False,False,FALLBACK_KE_AR5_UNIVARIAT (Jangan Klaim Sinyal...,Univariate AR(5),"Domino mengalahkan Naive (+1.04%), tetapi kala...",Tambahkan fitur eksogen curah hujan BMKG / pas...
2,cat_2,Daging Ayam,0.005470,0.005923,0.005501,7.13,6.72,False,False,FALLBACK_KE_NAIVE_RANDOM_WALK (Sinyal Sangat S...,Naive Last Price (Random Walk),"Domino menang telak atas AR(5) (+6.72%), namun...",Gunakan harga hari kerja terakhir atau rata-ra...
3,cat_3,Daging Sapi,0.002475,0.002816,0.002469,12.32,10.00,True,True,DEPLOY_DOMINO_ARX5 (Konsisten Menang di Raw & ...,Domino ARX(5) Ridge (Seasonal-Adjusted),Mengalahkan Naive dan AR(5) di kedua rezim (Sk...,Pertahankan di produksi; pantau kestabilan koe...
4,cat_4,Telur Ayam,0.002770,0.003001,0.002802,6.62,7.04,False,False,FALLBACK_KE_NAIVE_RANDOM_WALK (Sinyal Sangat S...,Naive Last Price (Random Walk),"Domino menang telak atas AR(5) (+7.04%), namun...",Gunakan harga hari kerja terakhir atau rata-ra...
5,cat_5,Bawang Merah,0.008892,0.007515,0.008469,-12.69,-7.84,False,False,FALLBACK_KE_AR5_UNIVARIAT (Jangan Klaim Sinyal...,Univariate AR(5),"Domino mengalahkan Naive (+4.76%), tetapi kala...",Tambahkan fitur eksogen curah hujan BMKG / pas...
6,cat_6,Bawang Putih,0.004568,0.004448,0.004539,-2.04,-0.93,False,False,FALLBACK_KE_AR5_UNIVARIAT (Jangan Klaim Sinyal...,Univariate AR(5),"Domino mengalahkan Naive (+0.64%), tetapi kala...",Tambahkan fitur eksogen curah hujan BMKG / pas...
7,cat_7,Cabai Merah,0.022205,0.020782,0.021367,-2.82,-3.39,False,False,FALLBACK_KE_AR5_UNIVARIAT (Jangan Klaim Sinyal...,Univariate AR(5),"Domino mengalahkan Naive (+3.77%), tetapi kala...",Tambahkan fitur eksogen curah hujan BMKG / pas...
8,cat_8,Cabai Rawit,0.019703,0.017318,0.018505,-6.85,-6.96,False,False,FALLBACK_KE_AR5_UNIVARIAT (Jangan Klaim Sinyal...,Univariate AR(5),"Domino mengalahkan Naive (+6.08%), tetapi kala...",Tambahkan fitur eksogen curah hujan BMKG / pas...
9,cat_9,Minyak Goreng,0.001581,0.001792,0.001551,13.47,14.79,True,False,DEPLOY_DOMINO_ARX5_RAW (Menang di Return Mentah),Domino ARX(5) Ridge (Raw Log-Return),"Unggul signifikan atas AR(5) (+13.47%, DM p=0....",Gunakan varian Raw Log-Return untuk prakiraan ...


### Q&A
- **Apakah pola efek domino harga layak dijadikan satu model tunggal bagi seluruh komoditas?** Tidak. Pengujian *walk-forward out-of-sample* sepanjang `296` hari kerja (`Hari [500:796]`) membuktikan bahwa model **Domino ARX(5) Ridge** memberikan peningkatan akurasi yang konsisten dan signifikan secara statistik pada **3 komoditas** (**Daging Sapi, Beras,** dan **Minyak Goreng**), sedangkan komoditas hortikultura bergejolak tinggi (**Cabai Merah, Cabai Rawit, Bawang Merah, Bawang Putih, Gula Pasir**) lebih akurat diprediksi oleh **AR(5) univariat**, serta **Daging Ayam** dan **Telur Ayam** paling rapat mengikuti **Naive Last Price**.
- **Bagaimana kualitas model rekomendasi bahan resep dan akar kelemahannya?** Pada prediksi bahan pelengkap (*hide-one-ingredient*, `3.459` resep uji), metode **Conditional Co-occurrence PMI** mencapai akurasi Top-5 **`45,50%`** (`CI95%: 43,85%–47,17%`) dan mengungguli Popularity baseline (`40,88%`, `CI95%: 39,25%–42,53%`). Namun pada substitusi lintas-bahan (`25` acuan kuliner), `Word2Vec` baru meraih `Recall@5 = 24,0% (6/25)`, `MRR@10 = 0,2100`, dan `NDCG@5 = 9,37%`.

### Data Analysis Key Findings
- **Dispersi Volatilitas (`eda_prices_stats.csv`):** Cabai Rawit (`CV = 20,80%`, std return `2,19%/hari`) dan Cabai Merah (`CV = 19,95%`, std return `2,56%/hari`) bergejolak hampir 7 kali lebih besar dibanding Beras (`CV = 2,96%`).
- **Validitas Statistik & Anti-Kebetulan (`M2`):** Uji permutasi *circular-shift* 20× menghasilkan rata-rata sisi palsu hanya **`3,44%`** (`3,1/90`, di bawah batas FDR `5%`), dan seluruh **`50/50` (`100%`)** hubungan Granger bertahan setelah efek musiman dan Hari Raya diisolasi.
- **Taksonomi Error Substitusi (`vp3_error_taxonomy.csv`):** Dari `19` kasus meleset di Top-5, **`10` kasus** berasal dari varian bentuk/olahan bahan yang sama (mis. `santan -> santan dari kelapa`, `gula merah -> blok gula merah`) dan **`9` kasus** berasal dari tetangga konteks masakan serumpun atau token korpus pendek (`miri`, `di`).
- **Efek Pareto Jembatan Harga (`M4`):** Pemetaan 10 komoditas PIHPS ke 14 bahan kosakata (`1,05%`) mencakup **`27,74%` (`217.380 / 783.564`)** seluruh kemunculan bahan dalam resep Nusantara.

### Insights or Next Steps
- **Kebijakan Model Hibrida Per Komoditas:** Aktifkan sinyal efek domino (`Domino ARX(5) Ridge`) khusus pada **Daging Sapi, Beras,** dan **Minyak Goreng** sesuai `model_review_decisions.csv`; untuk kelompok cabai dan bawang gunakan `AR(5)` univariat sembari menambahkan kovariat cuaca/curah hujan BMKG pada iterasi riset berikutnya.
- **Deduplikasi Morfologi Kandidat Substitusi:** Tambahkan lapisan pengelompokan induk bahan (*lemma deduplication*) di atas keluaran `Word2Vec` untuk menyaring varian bahan identik sebelum menampilkannya sebagai alternatif lintas bahan.